In [41]:
import pandas as pd
from minsearch import Index
from openai import OpenAI
from dotenv import load_dotenv

# Umgebung & Client
load_dotenv()
openai_client = OpenAI()

# Daten laden & Index einmalig bauen
df = pd.read_csv("data/chemical_contracts.csv")
documents = df.to_dict(orient="records")

index = Index(
    text_fields=["customer_name", "product_name", "force_majeure_clause"],
    keyword_fields=["contract_id", "currency"]
)
index.fit(documents)

In [53]:
def parse_question_to_filters(query):
    parser_prompt = f"""
You are an advanced supply chain data analyst. Convert the user question into a JSON object containing a list of conditions and a logic operator.
CRITICAL: Pay extreme attention to exact numbers and do not alter or drop digits (e.g., 5000 must remain 5000, not 500).

Available columns: contract_id, customer_name, product_name, base_price, currency, 
energy_adder_percentage, raw_material_adder_percentage, min_monthly_volume_tons, 
max_monthly_volume_tons, max_transport_duration_days, payment_terms_days, 
min_shelf_life_days, demurrage_days_included, breach_penalty_amount, force_majeure_clause.

Operators: "__le", "__ge", "__contains", or exact match (use null or "" for op if exact match).
Structure required:
{{
  "logic": "AND" (or "OR"),
  "conditions": [
    {{"col": "column_name", "op": "__le", "val": value}},
    ...
  ]
}}
If no conditions apply, return {{"logic": "AND", "conditions": []}}.

User Question: {query}
""".strip()

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": parser_prompt}],
        temperature=0
    )
    content = response.choices[0].message.content.strip()
    if content.startswith("```"):
        content = content.split("```")[1]
        if content.startswith("json"):
            content = content[4:]
    try:
        return json.loads(content.strip())
    except:
        return {"logic": "AND", "conditions": []}

In [54]:
import pandas as pd
import json

# 1. Der erweiterte Pandas-Filter (Versteht jetzt auch ODER)
def universal_contract_filter(conditions, logic="AND"):
    """
    conditions: Liste von Bedingungen (z.B. [{'col': 'base_price', 'op': '__le', 'val': 500}, ...])
    logic: "AND" oder "OR" für die Verknüpfung der Bedingungen
    """
    if not conditions:
        return df.copy()
    
    filtered_list = []
    
    for cond in conditions:
        col = cond["col"]
        op = cond["op"]
        val = cond["val"]
        
        if op == "__le":
            sub_df = df[df[col] <= val]
        elif op == "__ge":
            sub_df = df[df[col] >= val]
        elif op == "__contains":
            sub_df = df[df[col].astype(str).str.contains(val, case=False, na=False)]
        else:  # Exakter Match
            sub_df = df[df[col] == val]
            
        filtered_list.append(sub_df)
    
    if not filtered_list:
        return df.copy()
        
    # Wenn logic "AND" ist, nehmen wir den Schnittpunkt (Intersections)
    if logic.upper() == "AND":
        result = filtered_list[0]
        for sub_df in filtered_list[1:]:
            result = pd.merge(result, sub_df, how='inner')
        return result
    
    # Wenn logic "OR" ist, kombinieren wir alle (Unions)
    elif logic.upper() == "OR":
        result = pd.concat(filtered_list).drop_duplicates()
        return result
        
    return df.copy()

In [57]:
def smart_query(query):
    parsed = parse_question_to_filters(query)
    conditions = parsed.get("conditions", [])
    logic = parsed.get("logic", "AND")
    
    if conditions:
        print(f"[Router] Nutze Pandas-Filter mit Logik '{logic}' und Bedingungen: {conditions}")
        result_df = universal_contract_filter(conditions=conditions, logic=logic)
        
        if result_df.empty:
            return "No contracts match your criteria."
            
        # ==========================================
        # HIER MÜSSEN WIR DEN PROMPT ANPASSEN:
        # ==========================================
        summary_prompt = f"""
Here are the contracts matching the user's filter criteria:
{result_df.to_string()}

Based on these results, answer the user's question concisely: {query}
        """.strip()
        
        return llm(summary_prompt)
    else:
        if "lowest" in query.lower() or "highest" in query.lower():
            lowest_vol = df.loc[df['min_monthly_volume_tons'].idxmin()]
            return f"Lowest volume: {lowest_vol['contract_id']} ({lowest_vol['customer_name']})"
        return rag(query)

In [59]:
question = "Which customer has a minimum volume lower than 5000MT or a penalty higher than 1000?"

answer = smart_query(question)

print("Antwort:\n", answer)

[Router] Nutze Pandas-Filter mit Logik 'OR' und Bedingungen: [{'col': 'min_monthly_volume_tons', 'op': '__le', 'val': 5000}, {'col': 'breach_penalty_amount', 'op': '__ge', 'val': 1000}]
Antwort:
 The customers with a minimum volume lower than 5000 MT or a penalty higher than 1000 are:

1. **NovaChem Solutions** - Minimum Volume: 50.0 tons, Penalty: 15,000.0
2. **Green Planet Energy** - Minimum Volume: 100.0 tons, Penalty: 12,000.0
3. **Panther Industries** - Minimum Volume: 200.0 tons, Penalty: 3,000.0
4. **Titan Chemicals** - Minimum Volume: 150.0 tons, Penalty: 7,000.0
5. **ChemXpert Designs** - Minimum Volume: 250.0 tons, Penalty: 6,000.0
6. **Reliable Chem Supply** - Minimum Volume: 50.0 tons, Penalty: 2,000.0
7. **Calcium Carbonate** - Minimum Volume: 200.0 tons, Penalty: 0.0 (though penalty is not greater than 1000, it meets the volume criteria)
8. **Quantum Solutions** - Minimum Volume: 100.0 tons, Penalty: 10,000.0
9. **Optimum Chemicals** - Minimum Volume: 250.0 tons, Penalty: